In [34]:
import os
from tqdm import tqdm

# ==============================
# CONFIG
# ==============================
ROOT = "/home/nguyen/Downloads/dataset_fold_8_1_1"

splits = ["train", "val", "test"]

def check_split(split):
    print(f"\n🔍 Checking split: {split}")

    img_dir = os.path.join(ROOT, "images", split)
    lbl_dir = os.path.join(ROOT, "labels", split)

    img_files = sorted([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
    lbl_files = sorted([f for f in os.listdir(lbl_dir) if f.endswith(".txt")])

    img_set = set([f.replace(".jpg", "") for f in img_files])
    lbl_set = set([f.replace(".txt", "") for f in lbl_files])

    # Check missing labels or images
    missing_labels = img_set - lbl_set
    missing_images = lbl_set - img_set

    print(f"📸 Total images: {len(img_files)}")
    print(f"📝 Total labels : {len(lbl_files)}")

    if len(missing_labels) == 0:
        print("✅ All images have matching labels")
    else:
        print(f"❌ Images missing labels: {len(missing_labels)}")
        for x in list(missing_labels)[:10]:
            print("   -", x)

    if len(missing_images) == 0:
        print("✅ All labels have matching images")
    else:
        print(f"❌ Labels missing images: {len(missing_images)}")
        for x in list(missing_images)[:10]:
            print("   -", x)

    # Check label file content validity
    print("\n🔎 Checking label file format...")
    bad_labels = []
    for lf in tqdm(lbl_files):
        lbl_path = os.path.join(lbl_dir, lf)
        with open(lbl_path, "r") as f:
            lines = f.read().strip().splitlines()
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    bad_labels.append(lf)
                    break
                try:
                    cls = int(parts[0])
                    vals = list(map(float, parts[1:]))
                except:
                    bad_labels.append(lf)
                    break

    if len(bad_labels) == 0:
        print("✅ All label files are valid")
    else:
        print(f"❌ Found {len(bad_labels)} corrupted label files, example:")
        print(bad_labels[:10])

for split in splits:
    check_split(split)

print("\n🎯 DONE! Dataset verification complete.")



🔍 Checking split: train
📸 Total images: 8000
📝 Total labels : 8000
✅ All images have matching labels
✅ All labels have matching images

🔎 Checking label file format...


100%|██████████| 8000/8000 [00:00<00:00, 116007.36it/s]


✅ All label files are valid

🔍 Checking split: val
📸 Total images: 1000
📝 Total labels : 1000
✅ All images have matching labels
✅ All labels have matching images

🔎 Checking label file format...


100%|██████████| 1000/1000 [00:00<00:00, 110790.43it/s]


✅ All label files are valid

🔍 Checking split: test
📸 Total images: 1000
📝 Total labels : 1000
✅ All images have matching labels
✅ All labels have matching images

🔎 Checking label file format...


100%|██████████| 1000/1000 [00:00<00:00, 98908.27it/s]

✅ All label files are valid

🎯 DONE! Dataset verification complete.


In [22]:
import os
import random
from tqdm import tqdm

# ==============================
# CONFIG
# ==============================
ROOT = r"/home/nguyen/Downloads/dataset_fold_8_1_1"
IMG_TRAIN = os.path.join(ROOT, "images/train")
LBL_TRAIN = os.path.join(ROOT, "labels/train")

KEEP_COUNT = 8000  # giữ lại 8000 ảnh + nhãn


# ==============================
# LOAD FILE LIST
# ==============================
images = sorted([f for f in os.listdir(IMG_TRAIN) if f.endswith(".jpg")])
labels = sorted([f for f in os.listdir(LBL_TRAIN) if f.endswith(".txt")])

print("📸 Train images available:", len(images))
print("📝 Train labels available :", len(labels))


# ==============================
# FIND MISMATCHES
# ==============================
img_names = set([f.replace(".jpg", "") for f in images])
lbl_names = set([f.replace(".txt", "") for f in labels])

missing_labels = img_names - lbl_names    # ảnh thiếu nhãn
missing_images = lbl_names - img_names    # nhãn thiếu ảnh

print("\n❌ Images missing labels:", len(missing_labels))
print("\n❌ Labels missing images:", len(missing_images))


# ==============================
# DELETE IMAGES WITHOUT LABELS
# ==============================
print("\n🧹 Deleting images WITHOUT labels...")
for fname in tqdm(missing_labels):
    img_path = os.path.join(IMG_TRAIN, fname + ".jpg")
    if os.path.exists(img_path):
        os.remove(img_path)

# ==============================
# DELETE LABELS WITHOUT IMAGES
# ==============================
print("\n🧹 Deleting labels WITHOUT images...")
for fname in tqdm(missing_images):
    lbl_path = os.path.join(LBL_TRAIN, fname + ".txt")
    if os.path.exists(lbl_path):
        os.remove(lbl_path)


# ==============================
# RELOAD LIST AFTER CLEANING
# ==============================
images = sorted([f for f in os.listdir(IMG_TRAIN) if f.endswith(".jpg")])
labels = sorted([f for f in os.listdir(LBL_TRAIN) if f.endswith(".txt")])

print("\n📸 Images after cleaning:", len(images))
print("📝 Labels after cleaning :", len(labels))


# ==============================
# IF MORE THAN 8000 → RANDOM REMOVE
# ==============================
if len(images) > KEEP_COUNT:
    print(f"\n⚠️ Train has {len(images)} images > 8000 → trimming to 8000...")

    remove_count = len(images) - KEEP_COUNT

    # chọn ngẫu nhiên ảnh cần xoá
    images_to_delete = random.sample(images, remove_count)

    for img in tqdm(images_to_delete):
        img_path = os.path.join(IMG_TRAIN, img)
        lbl_path = os.path.join(LBL_TRAIN, img.replace(".jpg", ".txt"))

        if os.path.exists(img_path):
            os.remove(img_path)
        if os.path.exists(lbl_path):
            os.remove(lbl_path)

# ==============================
# FINAL COUNTS
# ==============================
final_images = len([f for f in os.listdir(IMG_TRAIN) if f.endswith(".jpg")])
final_labels = len([f for f in os.listdir(LBL_TRAIN) if f.endswith(".txt")])

print("\n🎯 DONE! TRAIN DATA CLEANED.")
print(f"📸 Final train images: {final_images}")
print(f"📝 Final train labels: {final_labels}")


📸 Train images available: 8000
📝 Train labels available : 8000

❌ Images missing labels: 0

❌ Labels missing images: 0

🧹 Deleting images WITHOUT labels...


0it [00:00, ?it/s]


🧹 Deleting labels WITHOUT images...


0it [00:00, ?it/s]


📸 Images after cleaning: 8000
📝 Labels after cleaning : 8000

🎯 DONE! TRAIN DATA CLEANED.
📸 Final train images: 8000
📝 Final train labels: 8000


In [31]:
import os

# ==========================
# CONFIG — sửa đường dẫn tại đây
# ==========================
ROOT = r"/home/nguyen/Downloads/dataset_fold_8_1_1"

PATHS = {
    "train_images": os.path.join(ROOT, "images/train"),
    "val_images":   os.path.join(ROOT, "images/val"),
    "test_images":  os.path.join(ROOT, "images/test"),

    "train_labels": os.path.join(ROOT, "labels/train"),
    "val_labels":   os.path.join(ROOT, "labels/val"),
    "test_labels":  os.path.join(ROOT, "labels/test"),
}

# ==========================
# COUNT FUNCTION
# ==========================
def count_files(path, ext):
    return len([f for f in os.listdir(path) if f.endswith(ext)])

# ==========================
# MAIN
# ==========================
print("\n📊 DATASET SUMMARY\n")

for split in ["train", "val", "test"]:
    img_dir = PATHS[f"{split}_images"]
    lbl_dir = PATHS[f"{split}_labels"]

    img_count = count_files(img_dir, ".jpg")
    lbl_count = count_files(lbl_dir, ".txt")

    print(f"📁 {split.upper()}:")
    print(f"   🖼️ Images : {img_count}")
    print(f"   📝 Labels : {lbl_count}\n")



📊 DATASET SUMMARY

📁 TRAIN:
   🖼️ Images : 8000
   📝 Labels : 8000

📁 VAL:
   🖼️ Images : 1000
   📝 Labels : 1000

📁 TEST:
   🖼️ Images : 1000
   📝 Labels : 1000



In [30]:
import os
from tqdm import tqdm

# ==============================
# CONFIG
# ==============================
ROOT = r"/home/nguyen/Downloads/dataset_fold_8_1_1"

SPLITS = ["train", "val", "test"]


# ==============================
# Function kiểm tra từng split
# ==============================
def check_split(split_name):
    print(f"\n🔍 Checking split: {split_name}")

    img_dir = os.path.join(ROOT, "images", split_name)
    lbl_dir = os.path.join(ROOT, "labels", split_name)

    images = sorted([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
    labels = sorted([f for f in os.listdir(lbl_dir) if f.endswith(".txt")])

    print(f"📸 Total images: {len(images)}")
    print(f"📝 Total labels : {len(labels)}")

    img_set = set([i.replace(".jpg", "") for i in images])
    lbl_set = set([l.replace(".txt", "") for l in labels])

    missing_labels = img_set - lbl_set
    missing_images = lbl_set - img_set

    # ---------- PRINT PROBLEMS ----------
    if missing_labels:
        print(f"❌ Images missing labels: {len(missing_labels)}")
        for name in list(missing_labels)[:10]:  # in tối đa 10 để tránh quá dài
            print(f"   - {name} (image only)")
    else:
        print("✅ All images have matching labels")

    if missing_images:
        print(f"❌ Labels missing images: {len(missing_images)}")
        for name in list(missing_images)[:10]:
            print(f"   - {name} (label only)")
    else:
        print("✅ All labels have matching images")

    # ---------- CHECK LABEL FORMAT ----------
    print("\n🔎 Checking label file format...")
    error_count = 0

    for lbl in tqdm(labels):
        path = os.path.join(lbl_dir, lbl)
        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:            # YOLO format = 5 fields
                    print(f"❌ Bad format in {lbl}: {line}")
                    error_count += 1
                    break

                try:
                    cls, cx, cy, w, h = map(float, parts)
                    # check giá trị hợp lệ
                    if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                        print(f"❌ Invalid values in {lbl}: {line}")
                        error_count += 1
                        break
                except:
                    print(f"❌ Non-numeric values in {lbl}: {line}")
                    error_count += 1
                    break

    if error_count == 0:
        print(f"✅ All labels in '{split_name}' are valid")
    else:
        print(f"⚠️ Found {error_count} label formatting errors in {split_name}")

    print("\n------------------------------------------")


# ==============================
# RUN CHECK FOR ALL SPLITS
# ==============================
for split in SPLITS:
    check_split(split)

print("\n🎯 Dataset verification complete.")



🔍 Checking split: train
📸 Total images: 8000
📝 Total labels : 8000
✅ All images have matching labels
✅ All labels have matching images

🔎 Checking label file format...


100%|██████████| 8000/8000 [00:00<00:00, 121065.64it/s]


✅ All labels in 'train' are valid

------------------------------------------

🔍 Checking split: val
📸 Total images: 1000
📝 Total labels : 1000
✅ All images have matching labels
✅ All labels have matching images

🔎 Checking label file format...


100%|██████████| 1000/1000 [00:00<00:00, 120132.44it/s]


✅ All labels in 'val' are valid

------------------------------------------

🔍 Checking split: test
📸 Total images: 1000
📝 Total labels : 1000
✅ All images have matching labels
✅ All labels have matching images

🔎 Checking label file format...


100%|██████████| 1000/1000 [00:00<00:00, 121524.71it/s]

✅ All labels in 'test' are valid

------------------------------------------

🎯 Dataset verification complete.


In [28]:
import os

# ==============================
# CONFIG
# ==============================
ROOT = r"/home/nguyen/Downloads/dataset_fold_8_1_1"

def list_ids(folder):
    """Trả về set chứa ID (tên file không có .jpg/.txt)."""
    imgs = os.listdir(folder)
    ids = set([f.replace(".jpg", "").replace(".txt", "") for f in imgs])
    return ids


# ==============================
# LOAD IMAGE IDs
# ==============================
train_ids = list_ids(os.path.join(ROOT, "images/train"))
val_ids   = list_ids(os.path.join(ROOT, "images/val"))
test_ids  = list_ids(os.path.join(ROOT, "images/test"))

print("📸 Counts:")
print(f"   Train: {len(train_ids)}")
print(f"   Val  : {len(val_ids)}")
print(f"   Test : {len(test_ids)}\n")


# ==============================
# CHECK DUPLICATES
# ==============================
dup_train_val  = train_ids & val_ids
dup_train_test = train_ids & test_ids
dup_val_test   = val_ids & test_ids

def report(name, dup_set):
    if len(dup_set) == 0:
        print(f"✅ Không có ảnh bị trùng giữa {name}")
    else:
        print(f"❌ PHÁT HIỆN {len(dup_set)} ảnh bị trùng giữa {name}!")
        print("   Ví dụ:", list(dup_set)[:10])


print("🔎 Checking duplicates between splits...\n")

report("TRAIN ↔ VAL", dup_train_val)
report("TRAIN ↔ TEST", dup_train_test)
report("VAL ↔ TEST", dup_val_test)

print("\n🎯 DONE! Duplicate check complete.")


📸 Counts:
   Train: 8000
   Val  : 1000
   Test : 1000

🔎 Checking duplicates between splits...

✅ Không có ảnh bị trùng giữa TRAIN ↔ VAL
✅ Không có ảnh bị trùng giữa TRAIN ↔ TEST
✅ Không có ảnh bị trùng giữa VAL ↔ TEST

🎯 DONE! Duplicate check complete.


In [1]:
import cv2
from ultralytics import YOLO

def parse_gt_line(line):
    """
    Trả về list các bounding box (x1,y1,x2,y2)
    hoặc list rỗng nếu frame không có UAV.
    """
    if "detections:" not in line:
        return []

    parts = line.split("detections:")[1].strip()

    # không có bbox
    if parts == "" or parts == ",":
        return []

    # tách các bbox dạng (x1, y1, x2, y2)
    boxes = []
    items = parts.split("),")
    for item in items:
        item = item.replace("(", "").replace(")", "").strip()
        if item == "":
            continue
        nums = item.split(",")
        if len(nums) != 4:
            continue
        x1, y1, x2, y2 = map(int, nums)
        boxes.append([x1, y1, x2, y2])
    return boxes


def evaluate_uav_video(model_path, video_path, gt_path, conf=0.25):
    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)

    gt_lines = open(gt_path, "r").read().strip().splitlines()

    frame_idx = 0
    total_frames_with_uav = 0
    miss_detect_frames = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # frame index trong GT là 1-based
        if frame_idx >= len(gt_lines):
            break

        gt_line = gt_lines[frame_idx]
        gt_boxes = parse_gt_line(gt_line)

        has_gt = len(gt_boxes) > 0

        if has_gt:
            total_frames_with_uav += 1

        # Y: chỉ evaluate frame có UAV
        if has_gt:
            results = model(frame, conf=conf)[0]
            pred_boxes = results.boxes

            if len(pred_boxes) == 0:
                miss_detect_frames += 1

        frame_idx += 1

    cap.release()

    print("\n========== EVALUATION RESULT ==========")
    print(f"Tổng frame có UAV (GT): {total_frames_with_uav}")
    print(f"Frame YOLO KHÔNG detect được UAV: {miss_detect_frames}")
    if total_frames_with_uav > 0:
        print(f"Tỉ lệ miss: {miss_detect_frames / total_frames_with_uav * 100:.2f}%")
    print("=======================================\n")

    return total_frames_with_uav, miss_detect_frames


# ==========================
# CHẠY DEMO
# ==========================

model_path = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
video_path = "/home/nguyen/Downloads/Videos/Videos/Clip_1.mov"
gt_path = "/home/nguyen/Downloads/Video_Annotation/Video_Annotation/Clip_1_gt.txt"

evaluate_uav_video(model_path, video_path, gt_path)



0: 384x640 (no detections), 23.1ms
Speed: 2.0ms preprocess, 23.1ms inference, 21.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.7ms
Speed: 2.7ms preprocess, 13.7ms inference, 4.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.6ms
Speed: 2.9ms preprocess, 12.6ms inference, 4.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.6ms
Speed: 2.2ms preprocess, 12.6ms inference, 4.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.6ms
Speed: 2.5ms preprocess, 12.6ms inference, 4.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.6ms
Speed: 1.7ms preprocess, 12.6ms inference, 5.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.6ms
Speed: 2.6ms preprocess, 13.6ms inference, 5.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.6ms
Speed: 1.4ms preprocess, 12.6ms 

(297, 130)

In [5]:
import cv2
from ultralytics import YOLO

def parse_gt_line(line):
    """
    Trả về list các bounding box (x1,y1,x2,y2)
    hoặc list rỗng nếu frame không có UAV.
    """
    if "detections:" not in line:
        return []

    parts = line.split("detections:")[1].strip()

    # không có bbox
    if parts == "" or parts == ",":
        return []

    # tách các bbox dạng (x1, y1, x2, y2)
    boxes = []
    items = parts.split("),")
    for item in items:
        item = item.replace("(", "").replace(")", "").strip()
        if item == "":
            continue
        nums = item.split(",")
        if len(nums) != 4:
            continue
        x1, y1, x2, y2 = map(int, nums)
        boxes.append([x1, y1, x2, y2])
    return boxes


def evaluate_uav_video(model_path, video_path, gt_path, conf=0.25):
    model = YOLO(model_path)

    cap = cv2.VideoCapture(video_path)

    gt_lines = open(gt_path, "r").read().strip().splitlines()

    frame_idx = 0
    total_frames_with_uav = 0
    miss_detect_frames = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # frame index trong GT là 1-based
        if frame_idx >= len(gt_lines):
            break

        gt_line = gt_lines[frame_idx]
        gt_boxes = parse_gt_line(gt_line)

        has_gt = len(gt_boxes) > 0

        if has_gt:
            total_frames_with_uav += 1

        # Y: chỉ evaluate frame có UAV
        if has_gt:
            results = model(frame, conf=conf)[0]
            pred_boxes = results.boxes

            if len(pred_boxes) == 0:
                miss_detect_frames += 1

        frame_idx += 1

    cap.release()

    print("\n========== EVALUATION RESULT ==========")
    print(f"Tổng frame có UAV (GT): {total_frames_with_uav}")
    print(f"Frame YOLO KHÔNG detect được UAV: {miss_detect_frames}")
    if total_frames_with_uav > 0:
        print(f"Tỉ lệ miss: {miss_detect_frames / total_frames_with_uav * 100:.2f}%")
    print("=======================================\n")

    return total_frames_with_uav, miss_detect_frames


# ==========================
# CHẠY DEMO
# ==========================

model_path = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
video_path = "/home/nguyen/Downloads/Videos/Videos/Clip_50.mov"
gt_path = "/home/nguyen/Downloads/Video_Annotation/Video_Annotation/Clip_50_gt.txt"

evaluate_uav_video(model_path, video_path, gt_path)



0: 384x640 2 uavs, 19.5ms
Speed: 2.7ms preprocess, 19.5ms inference, 20.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 uav, 12.6ms
Speed: 2.5ms preprocess, 12.6ms inference, 22.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 uav, 12.5ms
Speed: 1.4ms preprocess, 12.5ms inference, 20.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 uav, 12.5ms
Speed: 2.6ms preprocess, 12.5ms inference, 22.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 uavs, 12.4ms
Speed: 2.4ms preprocess, 12.4ms inference, 34.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 uavs, 12.6ms
Speed: 2.7ms preprocess, 12.6ms inference, 38.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 uav, 12.5ms
Speed: 3.4ms preprocess, 12.5ms inference, 20.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 uavs, 12.5ms
Speed: 2.7ms preprocess, 12.5ms inference, 35.0ms postprocess per image at shape (1, 3, 384, 640)

0: 

(1521, 100)

In [3]:
from ultralytics import YOLO
import cv2
import os

# ==========================
# CONFIG
# ==========================
MODEL_PATH = r"/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"  # đường dẫn model bạn đã train
VIDEO_PATH = r"/home/nguyen/Downloads/Videos/Videos/Clip_50.mov"                      # video test tương ứng
GT_PATH    = r"/home/nguyen/Downloads/Video_Annotation/Video_Annotation/Clip_50_gt.txt"                   # file ground truth gốc

UAV_CLASS_ID = 0     # nếu bạn chỉ có 1 class UAV thì để 0
CONF_THRES   = 0.25  # ngưỡng confidence YOLO
IOU_THRES    = 0.5   # ngưỡng IoU để coi là "phát hiện trúng UAV"


# ==========================
# HÀM PARSE GROUND TRUTH
# ==========================
def load_gt_frames(gt_path):
    """
    Đọc file Clip_X_gt.txt theo format:
    time_layer: 1 detections: (ymin,xmin,ymax,xmax) (ymin,xmin,ymax,xmax) ...
    Trả về:
        gt_frames: dict[frame_id] = list[(x1,y1,x2,y2)]
    """
    gt_frames = {}

    if not os.path.exists(gt_path):
        raise FileNotFoundError(f"Không tìm thấy GT file: {gt_path}")

    with open(gt_path, "r") as f:
        for line in f:
            line = line.strip()
            if "detections:" not in line:
                continue

            parts = line.split("detections:")
            # phần trước 'detections:' chứa time_layer: N
            left = parts[0].strip()   # "time_layer: 25"
            right = parts[1].strip()  # "(ymin,xmin,ymax,xmax) ..."

            # lấy frame_id
            # left dạng: "time_layer: 25"
            try:
                frame_id = int(left.split(":")[1].strip())
            except:
                continue

            if right == "" or right == "()":
                # dòng này không có UAV
                continue

            # chuẩn hóa chuỗi box: "(), ()" -> tách theo ")"
            # mình dùng cách giống code convert label trước đây
            boxes_str = right.replace("),", ")|")
            box_tokens = boxes_str.split("|")

            for box in box_tokens:
                box = box.strip().strip("(),")
                if not box:
                    continue

                # GT dạng: ymin,xmin,ymax,xmax
                try:
                    ymin, xmin, ymax, xmax = map(int, box.split(","))
                except:
                    continue

                x1, y1, x2, y2 = xmin, ymin, xmax, ymax

                if frame_id not in gt_frames:
                    gt_frames[frame_id] = []
                gt_frames[frame_id].append((x1, y1, x2, y2))

    return gt_frames


# ==========================
# HÀM TÍNH IoU
# ==========================
def bbox_iou(box1, box2):
    """
    box1, box2: (x1,y1,x2,y2)
    Trả về IoU (float)
    """
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    ix1 = max(x1, x1g)
    iy1 = max(y1, y1g)
    ix2 = min(x2, x2g)
    iy2 = min(y2, y2g)

    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0

    inter = (ix2 - ix1) * (iy2 - iy1)
    a1 = (x2 - x1) * (y2 - y1)
    a2 = (x2g - x1g) * (y2g - y1g)
    union = a1 + a2 - inter + 1e-6
    return inter / union


# ==========================
# HÀM EVAL VIDEO
# ==========================
def eval_video(model_path, video_path, gt_path,
               uav_class_id=0,
               conf_thres=0.25,
               iou_thres=0.5):

    # 1. Load GT
    gt_frames = load_gt_frames(gt_path)
    total_uav_frames = len(gt_frames)  # số frame có ít nhất 1 UAV theo GT

    print(f"📄 Ground truth loaded: {len(gt_frames)} frames có UAV.")

    # 2. Load model
    print("📦 Loading YOLO model...")
    model = YOLO(model_path)
    print("✅ Model loaded.\n")

    # 3. Video loop
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")

    frame_idx = 0
    miss_frames = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # nếu frame này KHÔNG có UAV trong GT thì bỏ qua
        if frame_idx not in gt_frames:
            continue

        gt_boxes = gt_frames[frame_idx]

        # 4. YOLO inference trên frame
        results = model(frame, conf=conf_thres, verbose=False)[0]

        # Lọc bbox theo class UAV
        pred_boxes = []
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            if cls_id != uav_class_id:
                continue
            if conf < conf_thres:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            pred_boxes.append((x1, y1, x2, y2))

        # Nếu không có bbox nào -> chắc chắn miss
        if len(pred_boxes) == 0:
            miss_frames += 1
            continue

        # 5. Kiểm tra IoU: nếu không bbox nào match GT => tính là miss
        detected = False
        for g in gt_boxes:
            for p in pred_boxes:
                iou = bbox_iou(p, g)
                if iou >= iou_thres:
                    detected = True
                    break
            if detected:
                break

        if not detected:
            miss_frames += 1

    cap.release()

    print("\n================= KẾT QUẢ =================")
    print(f"🔢 Tổng số frame có UAV (theo GT): {total_uav_frames}")
    print(f"🚫 Số frame KHÔNG detect được UAV: {miss_frames}")
    if total_uav_frames > 0:
        print(f"📉 Tỉ lệ miss detection: {miss_frames/total_uav_frames*100:.2f}%")
    print("===========================================")

    return total_uav_frames, miss_frames


# ==========================
# CHẠY THỬ
# ==========================
if __name__ == "__main__":
    total_uav_frames, miss_frames = eval_video(
        MODEL_PATH,
        VIDEO_PATH,
        GT_PATH,
        uav_class_id=UAV_CLASS_ID,
        conf_thres=CONF_THRES,
        iou_thres=IOU_THRES
    )


📄 Ground truth loaded: 1521 frames có UAV.
📦 Loading YOLO model...
✅ Model loaded.


================= KẾT QUẢ =================
🔢 Tổng số frame có UAV (theo GT): 1521
🚫 Số frame KHÔNG detect được UAV: 148
📉 Tỉ lệ miss detection: 9.73%


In [8]:
from ultralytics import YOLO
import cv2
import os

# ==========================
# CONFIG
# ==========================
MODEL_PATH = r"/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"  # đường dẫn model bạn đã train
VIDEO_PATH = r"/home/nguyen/Downloads/Videos/Videos/Clip_14.mov"                      # video test tương ứng
GT_PATH    = r"/home/nguyen/Downloads/Video_Annotation/Video_Annotation/Clip_14_gt.txt"                   # file ground truth gốc

UAV_CLASS_ID = 0     # nếu bạn chỉ có 1 class UAV thì để 0
CONF_THRES   = 0.25  # ngưỡng confidence YOLO
IOU_THRES    = 0.5   # ngưỡng IoU để coi là "phát hiện trúng UAV"


# ==========================
# HÀM PARSE GROUND TRUTH
# ==========================
def load_gt_frames(gt_path):
    """
    Đọc file Clip_X_gt.txt theo format:
    time_layer: 1 detections: (ymin,xmin,ymax,xmax) (ymin,xmin,ymax,xmax) ...
    Trả về:
        gt_frames: dict[frame_id] = list[(x1,y1,x2,y2)]
    """
    gt_frames = {}

    if not os.path.exists(gt_path):
        raise FileNotFoundError(f"Không tìm thấy GT file: {gt_path}")

    with open(gt_path, "r") as f:
        for line in f:
            line = line.strip()
            if "detections:" not in line:
                continue

            parts = line.split("detections:")
            # phần trước 'detections:' chứa time_layer: N
            left = parts[0].strip()   # "time_layer: 25"
            right = parts[1].strip()  # "(ymin,xmin,ymax,xmax) ..."

            # lấy frame_id
            # left dạng: "time_layer: 25"
            try:
                frame_id = int(left.split(":")[1].strip())
            except:
                continue

            if right == "" or right == "()":
                # dòng này không có UAV
                continue

            # chuẩn hóa chuỗi box: "(), ()" -> tách theo ")"
            # mình dùng cách giống code convert label trước đây
            boxes_str = right.replace("),", ")|")
            box_tokens = boxes_str.split("|")

            for box in box_tokens:
                box = box.strip().strip("(),")
                if not box:
                    continue

                # GT dạng: ymin,xmin,ymax,xmax
                try:
                    ymin, xmin, ymax, xmax = map(int, box.split(","))
                except:
                    continue

                x1, y1, x2, y2 = xmin, ymin, xmax, ymax

                if frame_id not in gt_frames:
                    gt_frames[frame_id] = []
                gt_frames[frame_id].append((x1, y1, x2, y2))

    return gt_frames


# ==========================
# HÀM TÍNH IoU
# ==========================
def bbox_iou(box1, box2):
    """
    box1, box2: (x1,y1,x2,y2)
    Trả về IoU (float)
    """
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    ix1 = max(x1, x1g)
    iy1 = max(y1, y1g)
    ix2 = min(x2, x2g)
    iy2 = min(y2, y2g)

    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0

    inter = (ix2 - ix1) * (iy2 - iy1)
    a1 = (x2 - x1) * (y2 - y1)
    a2 = (x2g - x1g) * (y2g - y1g)
    union = a1 + a2 - inter + 1e-6
    return inter / union


# ==========================
# HÀM EVAL VIDEO
# ==========================
def eval_video(model_path, video_path, gt_path,
               uav_class_id=0,
               conf_thres=0.25,
               iou_thres=0.5):

    # 1. Load GT
    gt_frames = load_gt_frames(gt_path)
    total_uav_frames = len(gt_frames)  # số frame có ít nhất 1 UAV theo GT

    print(f"📄 Ground truth loaded: {len(gt_frames)} frames có UAV.")

    # 2. Load model
    print("📦 Loading YOLO model...")
    model = YOLO(model_path)
    print("✅ Model loaded.\n")

    # 3. Video loop
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")

    frame_idx = 0
    miss_frames = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # nếu frame này KHÔNG có UAV trong GT thì bỏ qua
        if frame_idx not in gt_frames:
            continue

        gt_boxes = gt_frames[frame_idx]

        # 4. YOLO inference trên frame
        results = model(frame, conf=conf_thres, verbose=False)[0]

        # Lọc bbox theo class UAV
        pred_boxes = []
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            if cls_id != uav_class_id:
                continue
            if conf < conf_thres:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            pred_boxes.append((x1, y1, x2, y2))

        # Nếu không có bbox nào -> chắc chắn miss
        if len(pred_boxes) == 0:
            miss_frames += 1
            continue

        # 5. Kiểm tra IoU: nếu không bbox nào match GT => tính là miss
        detected = False
        for g in gt_boxes:
            for p in pred_boxes:
                iou = bbox_iou(p, g)
                if iou >= iou_thres:
                    detected = True
                    break
            if detected:
                break

        if not detected:
            miss_frames += 1

    cap.release()

    print("\n================= KẾT QUẢ =================")
    print(f"🔢 Tổng số frame có UAV (theo GT): {total_uav_frames}")
    print(f"🚫 Số frame KHÔNG detect được UAV: {miss_frames}")
    if total_uav_frames > 0:
        print(f"📉 Tỉ lệ miss detection: {miss_frames/total_uav_frames*100:.2f}%")
    print("===========================================")

    return total_uav_frames, miss_frames


# ==========================
# CHẠY THỬ
# ==========================
if __name__ == "__main__":
    total_uav_frames, miss_frames = eval_video(
        MODEL_PATH,
        VIDEO_PATH,
        GT_PATH,
        uav_class_id=UAV_CLASS_ID,
        conf_thres=CONF_THRES,
        iou_thres=IOU_THRES
    )


📄 Ground truth loaded: 459 frames có UAV.
📦 Loading YOLO model...
✅ Model loaded.


================= KẾT QUẢ =================
🔢 Tổng số frame có UAV (theo GT): 459
🚫 Số frame KHÔNG detect được UAV: 423
📉 Tỉ lệ miss detection: 92.16%


In [7]:
import cv2
from ultralytics import YOLO

# ==========================
# CONFIG
# ==========================
MODEL_PATH = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
VIDEO_PATH = "/home/nguyen/Downloads/Videos/Videos/Clip_14.mov"
GT_PATH    = "/home/nguyen/Downloads/Video_Annotation/Video_Annotation/Clip_14_gt.txt"

UAV_CLASS_ID = 0      # ID class UAV trong YOLO (thường là 0 nếu 1 lớp)
CONF_THRES   = 0.25   # Ngưỡng confidence YOLO
IOU_THRES    = 0.5    # Ngưỡng IoU để tính là phát hiện đúng UAV


# ==========================
# PARSE 1 DÒNG GT
# ==========================
def parse_gt_line(line):
    """
    Đọc 1 dòng GT dạng:
      time_layer: 1 detections: (ymin,xmin,ymax,xmax) (ymin,xmin,ymax,xmax) ...
    Trả về list các box [x1, y1, x2, y2] (tọa độ pixel).
    """
    line = line.strip()
    if "detections:" not in line:
        return []

    right = line.split("detections:")[1].strip()
    if right == "" or right == "()" or right == ",":
        return []

    boxes = []
    # Tách các bbox: "(...),(...)" -> "()", "()" ...
    tokens = right.replace("),", ")|").split("|")
    for t in tokens:
        t = t.strip().strip("(),")
        if not t:
            continue
        nums = t.split(",")
        if len(nums) != 4:
            continue

        # GT gốc: (ymin, xmin, ymax, xmax)
        ymin, xmin, ymax, xmax = map(int, nums)
        x1, y1, x2, y2 = xmin, ymin, xmax, ymax
        boxes.append([x1, y1, x2, y2])

    return boxes


# ==========================
# HÀM TÍNH IoU GIỮA 2 BOX
# ==========================
def bbox_iou(box1, box2):
    """
    box1, box2: [x1,y1,x2,y2]
    """
    x1, y1, x2, y2 = box1
    gx1, gy1, gx2, gy2 = box2

    ix1 = max(x1, gx1)
    iy1 = max(y1, gy1)
    ix2 = min(x2, gx2)
    iy2 = min(y2, gy2)

    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0

    inter = (ix2 - ix1) * (iy2 - iy1)
    a1 = (x2 - x1) * (y2 - y1)
    a2 = (gx2 - gx1) * (gy2 - gy1)
    union = a1 + a2 - inter + 1e-6
    return inter / union


# ==========================
# HÀM EVALUATE TRÊN 1 VIDEO
# ==========================
def evaluate_uav_video(
    model_path,
    video_path,
    gt_path,
    uav_class_id=0,
    conf=0.25,
    iou_thres=0.5
):
    # 1. Load model YOLO
    print("📦 Loading YOLO model...")
    model = YOLO(model_path)
    print("✅ Model loaded.\n")

    # 2. Đọc toàn bộ GT lines
    with open(gt_path, "r") as f:
        gt_lines = [l.strip() for l in f.readlines() if l.strip()]

    # 3. Mở video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")

    frame_idx = 0  # 0-based, tương ứng time_layer = frame_idx + 1
    total_frames_with_uav = 0
    miss_no_pred = 0          # frame có UAV nhưng YOLO không ra bbox UAV nào
    miss_bad_iou = 0          # frame có bbox UAV nhưng không bbox nào trùng GT (IoU < iou_thres)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx >= len(gt_lines):
            # Hết GT rồi thì dừng, tránh out-of-range
            break

        # Lấy GT của frame hiện tại
        gt_boxes = parse_gt_line(gt_lines[frame_idx])
        has_gt = len(gt_boxes) > 0

        if has_gt:
            total_frames_with_uav += 1

            # 4. YOLO inference
            results = model(frame, conf=conf, verbose=False)[0]

            # 5. Lọc bbox theo class UAV
            pred_uav_boxes = []
            for box in results.boxes:
                cls_id = int(box.cls[0])
                score = float(box.conf[0])
                if cls_id != uav_class_id:
                    continue
                if score < conf:
                    continue
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                pred_uav_boxes.append([x1, y1, x2, y2])

            # Không có bbox UAV nào → miss kiểu 1
            if len(pred_uav_boxes) == 0:
                miss_no_pred += 1
            else:
                # Có bbox UAV, check IoU với GT
                detected = False
                for g in gt_boxes:
                    for p in pred_uav_boxes:
                        iou = bbox_iou(p, g)
                        if iou >= iou_thres:
                            detected = True
                            break
                    if detected:
                        break

                if not detected:
                    miss_bad_iou += 1

        # sang frame tiếp theo
        frame_idx += 1

    cap.release()

    total_miss = miss_no_pred + miss_bad_iou

    print("\n========== EVALUATION RESULT ==========")
    print(f"🔢 Tổng frame có UAV (GT): {total_frames_with_uav}")
    print(f"🚫 Miss do KHÔNG có bbox UAV nào: {miss_no_pred}")
    print(f"🚫 Miss do bbox UAV lệch (IoU < {iou_thres}): {miss_bad_iou}")
    print(f"📉 Tổng frame miss: {total_miss}")
    if total_frames_with_uav > 0:
        print(f"📉 Tỉ lệ miss: {total_miss / total_frames_with_uav * 100:.2f}%")
    print("=======================================\n")

    return total_frames_with_uav, miss_no_pred, miss_bad_iou


# ==========================
# CHẠY DEMO
# ==========================
if __name__ == "__main__":
    evaluate_uav_video(
        MODEL_PATH,
        VIDEO_PATH,
        GT_PATH,
        uav_class_id=UAV_CLASS_ID,
        conf=CONF_THRES,
        iou_thres=IOU_THRES
    )


📦 Loading YOLO model...
✅ Model loaded.


========== EVALUATION RESULT ==========
🔢 Tổng frame có UAV (GT): 459
🚫 Miss do KHÔNG có bbox UAV nào: 305
🚫 Miss do bbox UAV lệch (IoU < 0.5): 118
📉 Tổng frame miss: 423
📉 Tỉ lệ miss: 92.16%



In [16]:
import cv2
from ultralytics import YOLO
import os
import numpy as np

# ==========================
# CONFIG
# ==========================
MODEL_PATH  = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
VIDEO_PATH  = "/home/nguyen/Downloads/Videos/Videos/Clip_2.mov"
GT_PATH     = "/home/nguyen/Downloads/Video_Annotation/Video_Annotation/Clip_2_gt.txt"
OUT_VIDEO   = "/home/nguyen/Downloads/Clip_2_pred_sliding.mp4"

UAV_CLASS_ID = 0      # ID class UAV trong YOLO (single class -> 0)
CONF_THRES   = 0.25   # Ngưỡng confidence YOLO
IOU_THRES    = 0.5    # Ngưỡng IoU để tính là detect đúng
TILE         = 640
OVERLAP      = 0.3
STEP         = int(TILE * (1 - OVERLAP))  # giống code slicing train (448)


# ==========================
# PARSE 1 DÒNG GT
# ==========================
def parse_gt_line(line):
    """
    Đọc 1 dòng GT dạng:
      time_layer: 1 detections: (ymin,xmin,ymax,xmax) (ymin,xmin,ymax,xmax) ...
    Trả về list các box [x1, y1, x2, y2] (pixel).
    """
    line = line.strip()
    if "detections:" not in line:
        return []

    right = line.split("detections:")[1].strip()
    if right == "" or right == "()" or right == ",":
        return []

    boxes = []
    tokens = right.replace("),", ")|").split("|")
    for t in tokens:
        t = t.strip().strip("(),")
        if not t:
            continue
        nums = t.split(",")
        if len(nums) != 4:
            continue

        # GT gốc: (ymin, xmin, ymax, xmax)
        ymin, xmin, ymax, xmax = map(int, nums)
        x1, y1, x2, y2 = xmin, ymin, xmax, ymax
        boxes.append([x1, y1, x2, y2])

    return boxes


# ==========================
# HÀM TÍNH IoU
# ==========================
def bbox_iou(box1, box2):
    """
    box1, box2: [x1,y1,x2,y2]
    """
    x1, y1, x2, y2 = box1
    gx1, gy1, gx2, gy2 = box2

    ix1 = max(x1, gx1)
    iy1 = max(y1, gy1)
    ix2 = min(x2, gx2)
    iy2 = min(y2, gy2)

    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0

    inter = (ix2 - ix1) * (iy2 - iy1)
    a1 = (x2 - x1) * (y2 - y1)
    a2 = (gx2 - gx1) * (gy2 - gy1)
    union = a1 + a2 - inter + 1e-6
    return inter / union


# ==========================
# SIMPLE NMS (TÙY CHỌN) - dùng để tránh vẽ quá nhiều bbox trùng
# ==========================
def nms(boxes, scores, iou_thres=0.5):
    """
    boxes: list [x1,y1,x2,y2]
    scores: list score tương ứng
    Trả về index các box giữ lại
    """
    if len(boxes) == 0:
        return []

    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)

        inter = w * h
        iou = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)

        inds = np.where(iou <= iou_thres)[0]
        order = order[inds + 1]

    return keep


# ==========================
# HÀM EVAL + SLIDING WINDOW
# ==========================
def evaluate_uav_video_sliding(
    model_path,
    video_path,
    gt_path,
    out_video_path,
    uav_class_id=0,
    conf=0.25,
    iou_thres=0.5,
):
    # 1. Load model YOLO
    print("📦 Loading YOLO model...")
    model = YOLO(model_path)
    print("✅ Model loaded.\n")

    # 2. Đọc toàn bộ GT
    with open(gt_path, "r") as f:
        gt_lines = [l.strip() for l in f.readlines() if l.strip()]

    # 3. Mở video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")

    fps  = cap.get(cv2.CAP_PROP_FPS)
    W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # VideoWriter để xuất video prediction
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_writer = cv2.VideoWriter(out_video_path, fourcc, fps, (W, H))

    frame_idx = 0  # 0-based, tương ứng time_layer = frame_idx + 1
    total_frames_with_uav = 0
    miss_no_pred = 0
    miss_bad_iou = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx >= len(gt_lines):
            # Hết GT rồi thì dừng để tránh out-of-range
            break

        gt_boxes = parse_gt_line(gt_lines[frame_idx])
        has_gt = len(gt_boxes) > 0

        # Nếu frame này có UAV theo GT
        if has_gt:
            total_frames_with_uav += 1

        # ===== SLIDING WINDOW 640x640 =====
        Hf, Wf = frame.shape[:2]
        xs = list(range(0, max(Wf - TILE, 0) + 1, STEP))
        ys = list(range(0, max(Hf - TILE, 0) + 1, STEP))

        # đảm bảo luôn có tile chạm mép phải / dưới
        if xs[-1] != Wf - TILE:
            xs.append(Wf - TILE)
        if ys[-1] != Hf - TILE:
            ys.append(Hf - TILE)

        frame_pred_boxes = []
        frame_pred_scores = []

        # Chỉ cần chạy YOLO khi frame có GT hoặc nếu bạn muốn cũng có thể detect full (ở đây mình eval trên frame có GT thôi)
        if has_gt:
            for ty in ys:
                for tx in xs:
                    tile = frame[ty:ty+TILE, tx:tx+TILE]
                    # tránh case tile ra ngoài biên
                    if tile.shape[0] != TILE or tile.shape[1] != TILE:
                        continue

                    # YOLO inference trên tile
                    results = model(tile, conf=conf, verbose=False)[0]

                    for box in results.boxes:
                        cls_id = int(box.cls[0])
                        score = float(box.conf[0])
                        if cls_id != uav_class_id:
                            continue
                        if score < conf:
                            continue

                        x1_t, y1_t, x2_t, y2_t = box.xyxy[0].tolist()
                        # chuyển về toạ độ full-frame
                        x1 = tx + x1_t
                        y1 = ty + y1_t
                        x2 = tx + x2_t
                        y2 = ty + y2_t

                        frame_pred_boxes.append([x1, y1, x2, y2])
                        frame_pred_scores.append(score)

        # Đánh giá miss / hit chỉ trên frame có GT
        if has_gt:
            if len(frame_pred_boxes) == 0:
                miss_no_pred += 1
            else:
                # NMS (tuỳ chọn, cho gọn)
                keep_inds = nms(frame_pred_boxes, frame_pred_scores, iou_thres=0.5)
                frame_pred_boxes_nms = [frame_pred_boxes[i] for i in keep_inds]

                detected = False
                for g in gt_boxes:
                    for p in frame_pred_boxes_nms:
                        iou = bbox_iou(p, g)
                        if iou >= iou_thres:
                            detected = True
                            break
                    if detected:
                        break

                if not detected:
                    miss_bad_iou += 1

        # ===== VẼ LÊN FRAME & GHI VIDEO =====
        # Vẽ GT (màu đỏ)
        for (x1, y1, x2, y2) in gt_boxes:
            cv2.rectangle(frame,
                          (int(x1), int(y1)),
                          (int(x2), int(y2)),
                          (0, 0, 255), 2)
            cv2.putText(frame, "GT",
                        (int(x1), int(y1)-5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 0, 255), 2)

        # Vẽ prediction (màu xanh lá)
        for (box, score) in zip(frame_pred_boxes, frame_pred_scores):
            x1, y1, x2, y2 = box
            cv2.rectangle(frame,
                          (int(x1), int(y1)),
                          (int(x2), int(y2)),
                          (0, 255, 0), 2)
            cv2.putText(frame, f"UAV {score:.2f}",
                        (int(x1), int(y1)-5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 255, 0), 2)

        # Ghi frame ra video output
        out_writer.write(frame)

        frame_idx += 1

    cap.release()
    out_writer.release()

    total_miss = miss_no_pred + miss_bad_iou

    print("\n========== EVALUATION RESULT (SLIDING WINDOW) ==========")
    print(f"🔢 Tổng frame có UAV (GT): {total_frames_with_uav}")
    print(f"🚫 Miss do KHÔNG có bbox UAV nào: {miss_no_pred}")
    print(f"🚫 Miss do bbox UAV lệch (IoU < {iou_thres}): {miss_bad_iou}")
    print(f"📉 Tổng frame miss: {total_miss}")
    if total_frames_with_uav > 0:
        print(f"📉 Tỉ lệ miss: {total_miss / total_frames_with_uav * 100:.2f}%")
    print(f"🎞 Video prediction đã lưu tại: {out_video_path}")
    print("========================================================\n")

    return total_frames_with_uav, miss_no_pred, miss_bad_iou


# ==========================
# MAIN
# ==========================
if __name__ == "__main__":
    evaluate_uav_video_sliding(
        MODEL_PATH,
        VIDEO_PATH,
        GT_PATH,
        OUT_VIDEO,
        uav_class_id=UAV_CLASS_ID,
        conf=CONF_THRES,
        iou_thres=IOU_THRES,
    )


📦 Loading YOLO model...
✅ Model loaded.


========== EVALUATION RESULT (SLIDING WINDOW) ==========
🔢 Tổng frame có UAV (GT): 1800
🚫 Miss do KHÔNG có bbox UAV nào: 465
🚫 Miss do bbox UAV lệch (IoU < 0.5): 477
📉 Tổng frame miss: 942
📉 Tỉ lệ miss: 52.33%
🎞 Video prediction đã lưu tại: /home/nguyen/Downloads/Clip_2_pred_sliding.mp4



In [ ]:
from ultralytics import YOLO
import cv2

MODEL_PATH = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
VIDEO_PATH = "/home/nguyen/Downloads/V_DRONE_069.mp4"
OUT_PATH   = "/home/nguyen/Downloads/V_DRONE_069_direct_pred.mp4"

CONF_THRES = 0.2
UAV_CLASS_ID = 0

def quick_test():
    model = YOLO(MODEL_PATH)

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError("Không mở được video")

    fps  = cap.get(cv2.CAP_PROP_FPS)
    W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUT_PATH, fourcc, fps, (W, H))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # YOLO sẽ tự resize về 640
        results = model(frame, conf=CONF_THRES, verbose=False)[0]

        num_det = 0
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            if cls_id != UAV_CLASS_ID:
                continue
            num_det += 1
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, f"UAV {conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        if frame_idx % 30 == 0:
            print(f"Frame {frame_idx}: {num_det} detections")

        writer.write(frame)

    cap.release()
    writer.release()
    print("✅ DONE. Video direct YOLO saved to:", OUT_PATH)

if __name__ == "__main__":
    quick_test()


Frame 30: 0 detections
Frame 60: 2 detections
Frame 90: 1 detections
Frame 120: 3 detections
Frame 150: 2 detections
Frame 180: 1 detections
Frame 210: 1 detections
Frame 240: 2 detections
Frame 270: 1 detections
Frame 300: 1 detections
✅ DONE. Video direct YOLO saved to: /home/nguyen/Downloads/V_DRONE_069_direct_pred.mp4


In [13]:
from ultralytics import YOLO
import cv2

MODEL_PATH = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
VIDEO_PATH = "/home/nguyen/Downloads/V_DRONE_112.mp4"
OUT_PATH   = "/home/nguyen/Downloads/V_DRONE_112_direct_pred.mp4"

CONF_THRES = 0.2
UAV_CLASS_ID = 0

def quick_test():
    model = YOLO(MODEL_PATH)

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError("Không mở được video")

    fps  = cap.get(cv2.CAP_PROP_FPS)
    W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUT_PATH, fourcc, fps, (W, H))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # YOLO sẽ tự resize về 640
        results = model(frame, conf=CONF_THRES, verbose=False)[0]

        num_det = 0
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            if cls_id != UAV_CLASS_ID:
                continue
            num_det += 1
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, f"UAV {conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        if frame_idx % 30 == 0:
            print(f"Frame {frame_idx}: {num_det} detections")

        writer.write(frame)

    cap.release()
    writer.release()
    print("✅ DONE. Video direct YOLO saved to:", OUT_PATH)

if __name__ == "__main__":
    quick_test()


Frame 30: 0 detections
Frame 60: 0 detections
Frame 90: 0 detections
Frame 120: 0 detections
Frame 150: 0 detections
Frame 180: 0 detections
Frame 210: 0 detections
Frame 240: 0 detections
Frame 270: 0 detections
Frame 300: 0 detections
✅ DONE. Video direct YOLO saved to: /home/nguyen/Downloads/V_DRONE_112_direct_pred.mp4


In [15]:
from ultralytics import YOLO
import cv2

MODEL_PATH = "/home/nguyen/Downloads/UAV_Slicing/final_dataset/runs_v8n_811/detect/yolov8_slicing/weights/best.pt"
VIDEO_PATH = "/home/nguyen/Downloads/V_DRONE_112.mp4"
OUT_PATH   = "/home/nguyen/Downloads/V_DRONE_112_debug_lowconf.mp4"

def debug_low_conf():
    model = YOLO(MODEL_PATH)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps  = cap.get(cv2.CAP_PROP_FPS)
    W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUT_PATH, fourcc, fps, (W, H))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # *** cố tình để conf rất thấp ***
        results = model(frame, conf=0.01, verbose=False)[0]

        num_det = len(results.boxes)
        print(f"Frame {frame_idx}: {num_det} predictions (ALL classes, ALL conf>=0.01)")

        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 1)
            cv2.putText(frame, f"{cls_id} {conf:.2f}", (x1, y1-3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0), 1)

        writer.write(frame)

    cap.release()
    writer.release()
    print("✅ Saved debug video to:", OUT_PATH)

if __name__ == "__main__":
    debug_low_conf()


Frame 1: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 2: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 3: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 4: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 5: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 6: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 7: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 8: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 9: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 10: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 11: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 12: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 13: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 14: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 15: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 16: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 17: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 18: 0 predictions (ALL classes, ALL conf>=0.01)
Frame 19: 0 predictions (ALL classes,